# Step 1a: History features (JDsearch personalised reranking, Group 9)

**Purpose.** Turn each user's past clicks, carts, follows and orders into features for reranking the candidate list of their final search. All features respect the **asymmetric retention windows** found in the interim report: the history keeps clicks for ~7 days, carts and follows for ~90 days, and orders for ~365 days.

**Pipeline in this notebook**

| Step | What it does | Output |
|---|---|---|
| 1 | Inspect the sample (`user_subset.parquet`) | – |
| 2 | Restore full histories (the sample had cut them to the last 200 actions) | `history_full.parquet` |
| 3 | Compute each past action's age; detect same-second checkout batches | `history_long.parquet` |
| 4 | User-level features (window counts, recency, in-session, last-7-day mix) | `history_user_features.parquet` |
| 5 | User × exact-product features | `history_candidate_features.parquet` |
| 6 | User × brand / shop / category features | `history_group_features.parquet` |
| 7 | Leakage check (motivates a ≤1 s guard used in steps 5–6) | – |
| 8 | Evidence that the features carry signal (train split only) | – |
| 9 | Reload checks and feature dictionary | `feature_dictionary.csv` |

**Join keys for teammates:** `session_id` (= row number in `user_subset.parquet`; do not re-sort that file) and `wid` (candidate product ID, int64). `label` (0 none, 1 click, 2 cart, 3 purchase) is the **target**, never a feature. Use the existing `split` column (train / val / test, by user).

**How to run.** Google Colab with standard RAM (~12.7 GB). Set `BASE` in the first code cell to the Drive folder containing `user_subset.parquet`, `user_behavior_data.txt` and `product_meta_data.txt`. Outputs are written to `BASE/step1a_history/`. A full *Restart and run all* takes roughly 15–25 minutes, mostly reading the two raw text files.

**Key results** (details under each step)
- The 200-action cut in the sample removed **50.7% of orders** but only **18.2% of clicks**, so full histories were restored.
- **53.6%** of orders are same-second checkout batches (3.35 M order rows = 2.08 M checkouts), so orders are counted as checkouts.
- Products the user touched before are engaged **24.9% vs 1.9%**. Previously **ordered** products are re-bought at **8.8%** (≈73× the baseline).
- Brand / shop / category history reaches far more candidates (20% / 7% / 37% of rows vs 0.8% for exact product). **Shop** history is the strongest of the three (≈4–5× engagement and purchase).

## Step 1: Inspect the sample
Load the group's sample (`user_subset.parquet`: 85,864 users, one final search each) and check its structure before building on it.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path('/content/drive/MyDrive/JDsearch/')
df = pd.read_parquet(BASE / 'user_subset.parquet')

print('Rows, columns:', df.shape)
print('Memory used: %.2f GB' % (df.memory_usage(deep=True).sum() / 1e9))
print()
print(df.dtypes)

Mounted at /content/drive
Rows, columns: (85864, 9)
Memory used: 0.37 GB

query                   object
candidate_wid_list      object
candidate_label_list    object
history_qry_list        object
history_wid_list        object
history_type_list       object
history_time_list       object
has_purchase              bool
split                   object
dtype: object


Look at one row. Lists are stored as `_`-joined text. Query terms are separated by an invisible `\x18` character (Colab shows it as a box), as documented in the dataset README.

In [2]:
row = df.iloc[0]
for col in df.columns:
    value = row[col]
    print(f'{col:30s} | {type(value).__name__:10s} | {str(value)[:120]}')

query                          | str        | 417104399485348760257419982968859485348760257419941047392989136485271452
candidate_wid_list             | str        | 66397137_84554371_34745767_44666310_54175196_8434849_94152752_74618574_60791847_77305831_31867013_72867003_18893425_3801
candidate_label_list           | str        | 2.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_
history_qry_list               | str        | 44967883_796388565763631260257419602574193795030948315893_-1_-1_563808948817664_563808948817664_-1_4124768018321
history_wid_list               | str        | 90584349_51004412_64885884_57738660_96865106_1632419_74472932_80030934_17734143_55519677_15026238_19099136_36676703_2358
history_type_list              | str        | ORD_ORD_ORD_ORD_ORD_ORD_ORD_ORD_ORD_ORD_ORD_ORD_ORD_ORD_ORD_ORD_ORD_ORD_ORD_ORD_ORD_ORD_ORD_ORD_ORD_CART_ORD_CART_CART_O
history_time_list              | str  

Check (a) whether histories were truncated, (b) that the time list is always one longer than the action list (its last value is the gap between the final action and the search), and (c) whether a train/val/test column exists.

In [3]:
def to_list(v):
    """Turn a value into a Python list, whether it's stored as '1_2_3' text or as a list."""
    if isinstance(v, str):
        return v.split('_') if v else []
    return list(v)

wid_col  = [c for c in df.columns if 'history_wid'  in c][0]
time_col = [c for c in df.columns if 'history_time' in c][0]

hist_len = df[wid_col].apply(lambda v: len(to_list(v)))
time_len = df[time_col].apply(lambda v: len(to_list(v)))

print('History length per user:')
print(hist_len.describe())
print()
print('Users with exactly 200 actions:', (hist_len == 200).sum())
print('Users with more than 200:      ', (hist_len > 200).sum())
print('Time list = history + 1 for all rows?', (time_len == hist_len + 1).mean() == 1.0)
print()
print('Columns that look like a split:', [c for c in df.columns if 'split' in c.lower()])

History length per user:
count    85864.000000
mean        97.050370
std         73.449807
min          1.000000
25%         29.000000
50%         78.000000
75%        188.000000
max        200.000000
Name: history_wid_list, dtype: float64

Users with exactly 200 actions: 20041
Users with more than 200:       0
Time list = history + 1 for all rows? True

Columns that look like a split: ['split']


> **Result.** 85,864 searches with a `split` column. There is **no ID column**, so `session_id` is defined as the row number. Histories **were truncated** to the last 200 actions: 20,041 users (23.3%) sit exactly at the cap. Time lists are consistent in every row.

## Step 2: Restore full histories
Truncating to the most recent 200 actions removes each heavy user's **oldest** actions. Clicks expire after ~7 days anyway, so the removed actions are mostly carts and orders. That distorts exactly the 90- and 365-day windows. We therefore re-attach each sampled user's complete history from the raw `user_behavior_data.txt`, matching searches by their candidate list.

In [4]:
hist_cols = ['history_qry_list', 'history_wid_list', 'history_type_list', 'history_time_list']

raw = pd.read_csv(BASE / 'user_behavior_data.txt', sep='\t', dtype=str,
                  usecols=['candidate_wid_list'] + hist_cols)

print('Raw rows:', len(raw))
print('Candidate lists that appear more than once:', raw['candidate_wid_list'].duplicated().sum())

Raw rows: 173831
Candidate lists that appear more than once: 29


In [5]:
df['session_id'] = np.arange(len(df))   # row number = session_id

raw_full = raw.rename(columns={c: c + '_full' for c in hist_cols})
merged = df[['session_id', 'candidate_wid_list'] + hist_cols].merge(
    raw_full, on='candidate_wid_list', how='left')

print('Rows after matching:', len(merged), '(should be', len(df), ')')
print('Searches with no match:', merged['history_wid_list_full'].isna().sum())

Rows after matching: 85887 (should be 85864 )
Searches with no match: 0


The raw file contains 29 repeated candidate lists, which creates 23 wrong extra matches. We keep only the raw row whose full history **ends with** the sample's history and has the **same gap to the search**, then check that each search is matched exactly once.

In [6]:
def subset_is_tail_of_full(row):
    full = row['history_wid_list_full'].split('_')
    sub  = row['history_wid_list'].split('_')
    return full[-len(sub):] == sub

tail_ok = merged.apply(subset_is_tail_of_full, axis=1)
last_gap_sub  = merged['history_time_list'].str.rsplit('_', n=1).str[-1]
last_gap_full = merged['history_time_list_full'].str.rsplit('_', n=1).str[-1]

print('Subset history = end of full history:', tail_ok.mean())
print('Same gap to the search:              ', (last_gap_sub == last_gap_full).mean())

Subset history = end of full history: 0.9997322062710305
Same gap to the search:               0.9997554926822453


In [8]:
good = merged[tail_ok & (last_gap_sub == last_gap_full)]
good = good.drop_duplicates('session_id')

print('Searches with a correct match:', len(good), 'of', len(df))
print('Each search appears once?     ', good['session_id'].is_unique)

Searches with a correct match: 85864 of 85864
Each search appears once?      True


How much did the 200-action cut remove, by action type? (Computed on the validated matches only.)

In [9]:
from collections import Counter

kept, full = Counter(), Counter()
for sub_types, full_types in zip(good['history_type_list'], good['history_type_list_full']):
    kept.update(sub_types.split('_'))
    full.update(full_types.split('_'))

for t in ['CLICK', 'CART', 'FLW', 'ORD']:
    print(f"{t:6s} full={full[t]:>10,}  kept={kept[t]:>10,}  dropped={1 - kept[t]/full[t]:.1%}")

CLICK  full= 4,076,215  kept= 3,333,876  dropped=18.2%
CART   full= 5,686,808  kept= 3,340,185  dropped=41.3%
FLW    full=    12,884  kept=     4,450  dropped=65.5%
ORD    full= 3,353,240  kept= 1,654,622  dropped=50.7%


> **Result.** All 85,864 searches matched exactly once. Truncation had removed **50.7% of orders**, 41.3% of carts and 65.5% of follows, but only **18.2% of clicks**. The restored full histories are used from here on.

Build the working table (the sample with full histories and `session_id`) and save it, so later steps never need to repeat the matching.

In [10]:
work = df.drop(columns=hist_cols).merge(
    good[['session_id'] + [c + '_full' for c in hist_cols]], on='session_id', how='left')
work = work.rename(columns={c + '_full': c for c in hist_cols})

OUT = BASE / 'step1a_history'
OUT.mkdir(exist_ok=True)
work.to_parquet(OUT / 'history_full.parquet', index=False)

print(work.shape)
print(work.columns.tolist())

del raw, raw_full, merged, good   # free up memory

(85864, 10)
['query', 'candidate_wid_list', 'candidate_label_list', 'has_purchase', 'split', 'session_id', 'history_qry_list', 'history_wid_list', 'history_type_list', 'history_time_list']


## Step 3: Action ages and checkout batches
### 3a. How long before the search did each action happen?
History only stores the **gap since the previous action**; the last gap is between the final action and the search. Each action's age is therefore the sum of all later gaps. Example: gaps `0_12_4_5` give ages 21, 9 and 5 seconds. We also reshape history into a **long table** (one row per past action), which makes window counts simple filters. `from_search` records whether the action came from a search (query ≠ −1).

In [11]:
s_ids, wid_parts, type_parts, age_parts, srch_parts = [], [], [], [], []

for sid, w, t, tm, q in zip(work['session_id'], work['history_wid_list'], work['history_type_list'],
                            work['history_time_list'], work['history_qry_list']):
    gaps = np.array(tm.split('_'), dtype=np.int64)     # the gaps
    age  = np.cumsum(gaps[::-1])[::-1][1:]              # add up the gaps from the end -> age of each action
    n = len(age)
    s_ids.append(np.full(n, sid, dtype=np.int32))
    wid_parts.append(np.array(w.split('_'), dtype=np.int64))
    type_parts.append(t.split('_'))
    age_parts.append(age)
    srch_parts.append(np.array([x != '-1' for x in q.split('_')]))   # did this action come from a search?

hist = pd.DataFrame({
    'session_id':  np.concatenate(s_ids),
    'wid':         np.concatenate(wid_parts),
    'type':        pd.Categorical(np.concatenate(type_parts)),
    'age_sec':     np.concatenate(age_parts),
    'from_search': np.concatenate(srch_parts),
})
hist['age_days'] = hist['age_sec'] / 86400

print(hist.shape)
print('Memory: %.2f GB' % (hist.memory_usage(deep=True).sum() / 1e9))
hist.head()

(13129147, 6)
Memory: 0.39 GB


,session_id,wid,type,age_sec,from_search,age_days
0,0,90584349,ORD,25738928,True,297.904259
1,0,51004412,ORD,25574622,True,296.002569
2,0,64885884,ORD,23113481,False,267.517141
3,0,57738660,ORD,18039314,False,208.788356
4,0,96865106,ORD,17792558,True,205.932384


Check: the ages should reproduce the interim report's retention windows. The interim used the **full** dataset (all 173,831 users), so small differences are sampling variation.

In [12]:
summary = hist.groupby('type', observed=True)['age_days'].describe(percentiles=[0.5, 0.999])
print(summary[['count', '50%', '99.9%', 'max']].round(1))

           count    50%  99.9%    max
type                                 
CART   5686808.0   38.6   89.5  160.1
CLICK  4076215.0    2.8    6.9    7.0
FLW      12884.0   40.5   89.5   89.9
ORD    3353240.0  154.0  364.0  365.0


> **Result.** 13,129,147 actions (exactly the full-history total from step 2). Median ages are clicks 2.8 d, carts 38.6 d, follows 40.5 d and orders 154.0 d (interim: 2.8 / 38.4 / 39.4 / 154.1). 99.9% of clicks are within 7 days, carts and follows within 90, and orders within 365. The order median matching the interim also confirms the restoration worked: truncated histories would have pulled it well down.

### 3b. Save the long table, then detect checkout batches
Several orders logged in the **same second** are one checkout. We count checkouts as distinct (session, timestamp) pairs among orders.

In [13]:
hist.to_parquet(OUT / 'history_long.parquet', index=False)
print('saved')

saved


In [14]:
orders = hist[hist['type'] == 'ORD']                     # keep only order rows
batch_size = orders.groupby(['session_id', 'age_sec'])['wid'].transform('size')   # how many orders share that exact second

print('Order rows:', len(orders))
print('Checkouts (same-second orders merged):', len(orders[['session_id', 'age_sec']].drop_duplicates()))
print('Share of orders in a same-second batch: %.1f%%' % (100 * (batch_size > 1).mean()))

Order rows: 3353240
Checkouts (same-second orders merged): 2084985
Share of orders in a same-second batch: 53.6%


> **Result.** 53.6% of orders belong to a same-second batch (interim, full data: 53.7%). 3,353,240 order rows correspond to 2,084,985 checkouts, so raw order counts overstate purchase occasions by ≈61%.

## Step 4: User-level features (one row per search)
### 4a. Counts within each retention window
Each action type is counted only within its own window (clicks 7 d, carts/follows 90 d, orders 365 d) and types are **never added together or divided by one another**. Mixing a 7-day click count with a 365-day order count is what made last year's `click_ratio` / `order_ratio` mostly measure "did the user visit this week". Orders are also counted as deduplicated checkouts and as distinct products.

In [15]:
WIN_DAYS = {'CLICK': 7, 'CART': 90, 'FLW': 90, 'ORD': 365}
TYPES = ['CLICK', 'CART', 'FLW', 'ORD']

# mark whether each action is inside its own type's window
hist['in_window'] = hist['age_days'] <= hist['type'].map(WIN_DAYS).astype('float32')
print('Actions outside their window (dropped from counts):')
print(hist.loc[~hist['in_window'], 'type'].value_counts())

inw = hist[hist['in_window']]
idx = pd.Index(work['session_id'], name='session_id')   # every search, in order

# count each type per search
cnt = (inw.groupby(['session_id', 'type'], observed=True).size()
          .unstack().reindex(index=idx, columns=TYPES).fillna(0))

# orders: merge same-second batches, and count distinct products
ordw = inw[inw['type'] == 'ORD']
n_checkouts    = ordw.drop_duplicates(['session_id', 'age_sec']).groupby('session_id').size().reindex(idx, fill_value=0)
n_ord_products = ordw.drop_duplicates(['session_id', 'wid']).groupby('session_id').size().reindex(idx, fill_value=0)

# put it all in one table, one row per search
uf = pd.DataFrame(index=idx)
uf['split']                 = work.set_index('session_id')['split']
uf['n_click_7d']            = cnt['CLICK'].astype('int32')
uf['n_cart_90d']            = cnt['CART'].astype('int32')
uf['n_follow_90d']          = cnt['FLW'].astype('int32')
uf['n_order_rows_365d']     = cnt['ORD'].astype('int32')
uf['n_checkouts_365d']      = n_checkouts.astype('int32')
uf['n_order_products_365d'] = n_ord_products.astype('int32')

print(uf.shape)
display(uf.head())
display(uf.drop(columns='split').describe().T[['mean', '50%', 'min', 'max']].round(1))

Actions outside their window (dropped from counts):
type
CART     622
CLICK      0
FLW        0
ORD        0
Name: count, dtype: int64
(85864, 7)


,split,n_click_7d,n_cart_90d,n_follow_90d,n_order_rows_365d,n_checkouts_365d,n_order_products_365d
session_id,,,,,,,
0,train,0,16,0,35,27,35
1,train,48,18,0,48,46,47
2,train,31,34,0,11,11,11
3,train,100,29,0,7,7,7
4,train,1,22,0,7,5,7


,mean,50%,min,max
n_click_7d,47.5,17.0,0.0,3085.0
n_cart_90d,66.2,29.0,0.0,3418.0
n_follow_90d,0.2,0.0,0.0,695.0
n_order_rows_365d,39.1,15.0,0.0,2962.0
n_checkouts_365d,24.3,11.0,0.0,1994.0
n_order_products_365d,35.5,14.0,0.0,2743.0


### 4b. Recency
Days since the last click, cart and order (each within its window), and since any action. If a type has no action in its window, recency is set to the window length ("at least this long ago") and a `has_*` flag is set to 0. `days_since_last_action` tells the model whether a user with no clicks was simply away (e.g. 28 days) or is active but doesn't click. `in_session` = 1 if the last action was within 30 minutes of the search (an ongoing visit; the 30-minute break comes from the interim's gap analysis).

In [16]:
# most recent action of each type, per search (smallest age = most recent)
rec = (inw.groupby(['session_id', 'type'], observed=True)['age_days'].min()
          .unstack().reindex(index=idx, columns=TYPES))

for t, name, w in [('CLICK', 'click', 7), ('CART', 'cart', 90), ('ORD', 'order', 365)]:
    uf[f'has_{name}']             = rec[t].notna().astype('int8')          # 1 = there was one in the window
    uf[f'days_since_last_{name}'] = rec[t].fillna(w).astype('float32')     # none -> window length

# most recent action of any type
last_age_sec = hist.groupby('session_id')['age_sec'].min().reindex(idx)
uf['days_since_last_action'] = (last_age_sec / 86400).astype('float32')
uf['in_session'] = (last_age_sec <= 30 * 60).astype('int8')                # last action within 30 min

new_cols = ['has_click', 'days_since_last_click', 'has_cart', 'days_since_last_cart',
            'has_order', 'days_since_last_order', 'days_since_last_action', 'in_session']
display(uf[new_cols].head().round(2))
display(uf[new_cols].describe().T[['mean', '50%', 'min', 'max']].round(2))

,has_click,days_since_last_click,has_cart,days_since_last_cart,has_order,days_since_last_order,days_since_last_action,in_session
session_id,,,,,,,,
0,0,7.00,1,28.00,1,44.84,28.00,0
1,1,0.01,1,0.01,1,5.21,0.01,1
2,1,1.25,1,2.04,1,2.04,1.25,0
3,1,1.86,1,1.86,1,1.93,1.86,0
4,1,0.00,1,7.90,1,56.02,0.00,1


,mean,50%,min,max
has_click,0.84,1.00,0.0,1.00
days_since_last_click,1.85,0.36,0.0,7.00
has_cart,0.96,1.00,0.0,1.00
days_since_last_cart,10.79,2.04,0.0,90.00
has_order,0.91,1.00,0.0,1.00
days_since_last_order,61.93,13.37,0.0,365.00
days_since_last_action,5.41,0.24,0.0,363.12
in_session,0.41,0.00,0.0,1.00


### 4c. Last-7-day mix, history length, and save
Within the last 7 days every action type is recorded, so the share of carts and orders there (`intent_share_7d`) is a fair replacement for whole-history ratios. `history_len_full` is the length of the restored history.

In [17]:
h7 = hist[hist['age_days'] <= 7]                              # actions in the last 7 days
cnt7 = (h7.groupby(['session_id', 'type'], observed=True).size()
           .unstack().reindex(index=idx, columns=TYPES).fillna(0))

uf['n_actions_7d']     = cnt7.sum(axis=1).astype('int32')
uf['intent_share_7d']  = ((cnt7['CART'] + cnt7['ORD']) / uf['n_actions_7d']).fillna(0).astype('float32')
uf['history_len_full'] = hist.groupby('session_id').size().reindex(idx, fill_value=0).astype('int32')

print('Shape:', uf.shape)
print('Missing values:', uf.isna().sum().sum())
display(uf[['n_actions_7d', 'intent_share_7d', 'history_len_full']].describe().T[['mean', '50%', 'min', 'max']].round(2))

uf.reset_index().to_parquet(OUT / 'history_user_features.parquet', index=False)
print('saved history_user_features.parquet')

Shape: (85864, 18)
Missing values: 0


,mean,50%,min,max
n_actions_7d,56.29,21.00,0.0,3129.0
intent_share_7d,0.17,0.12,0.0,1.0
history_len_full,152.91,78.00,1.0,4719.0


saved history_user_features.parquet


> **Result.** 85,864 × 18 user features (plus `session_id`), with no missing values. 16% of users have no click in the last 7 days, so last year's ratios would have mislabelled them as "non-clickers". `in_session` = 41% (interim: 41.5%). The restored mean history length is 152.9 actions (vs 97.1 truncated).
>
> **Note.** User-level features are identical for every candidate in a search's list, so on their own they cannot reorder it. They give the ranker context and support segment analysis. The ranking signal comes from the user × candidate features in steps 5–6.

## Step 5: User × exact-product features
### 5a. Candidate table (the "scoring sheet")
One row per (search, candidate product), with the interaction label (0–3) and split. The label is kept for training and evaluation only and is never used to build features.

In [18]:
c_sid, c_wid, c_lab = [], [], []
for sid, w, l in zip(work['session_id'], work['candidate_wid_list'], work['candidate_label_list']):
    wids = np.array(w.split('_'), dtype=np.int64)
    c_sid.append(np.full(len(wids), sid, dtype=np.int32))
    c_wid.append(wids)
    c_lab.append(np.array(l.split('_'), dtype=np.float32).astype(np.int8))   # '2.0' -> 2

cand = pd.DataFrame({
    'session_id': np.concatenate(c_sid),
    'wid':        np.concatenate(c_wid),
    'label':      np.concatenate(c_lab),
})
cand['split'] = cand['session_id'].map(uf['split'])

print('Rows:', len(cand))
print('Each (search, product) appears once?', not cand.duplicated(['session_id', 'wid']).any())
print('\nLabels:');  print(cand['label'].value_counts().sort_index())
print('\nSplit:');   print(cand['split'].value_counts())
cand.head()

Rows: 7665446
Each (search, product) appears once? True

Labels:
label
0    7505211
1     112099
2      37708
3      10428
Name: count, dtype: int64

Split:
split
train    5360982
val      1156791
test     1147673
Name: count, dtype: int64


,session_id,wid,label,split
0,0,66397137,2,train
1,0,84554371,0,train
2,0,34745767,0,train
3,0,44666310,0,train
4,0,54175196,0,train


### 5b. Has the user touched *this exact product* before?
For each (search, candidate): counts of past clicks, carts, follows and orders on that exact product (within each window), and days since the last click, cart, order and any action. Missing counts are 0; missing recency is the window length; `prod_seen_before` flags any past interaction.

For speed, we first keep only past actions on products that are in that user's current list. This cannot change any feature value, because actions on other products are irrelevant to "this exact product". The full history is kept, and all user-level features (step 4) used it.

**Leakage guard.** History actions **≤1 s before the search** are ignored. Step 7 shows that engaged candidates were ≈10× over-represented among products last touched within 1 s of the search, consistent with the current search's own actions being logged into history.

In [19]:
# 1. keep only past actions on products that are in that person's current list
#    (leakage guard: ignore actions <=1 s before the search; see step 7)

hp = inw[inw['age_sec'] > 1].merge(cand[['session_id', 'wid']], on=['session_id', 'wid'], how='inner')
print('Past actions on products in the current list:', len(hp))

# 2. per (search, product, type): how many times, and how recently
g = hp.groupby(['session_id', 'wid', 'type'], observed=True)['age_days']
n_by    = g.size().unstack().reindex(columns=TYPES).fillna(0)
last_by = g.min().unstack().reindex(columns=TYPES)

pf = pd.DataFrame(index=n_by.index)
pf['prod_n_click']  = n_by['CLICK']
pf['prod_n_cart']   = n_by['CART']
pf['prod_n_follow'] = n_by['FLW']
pf['prod_n_order']  = n_by['ORD']
pf['prod_days_since_click'] = last_by['CLICK']
pf['prod_days_since_cart']  = last_by['CART']
pf['prod_days_since_order'] = last_by['ORD']
pf['prod_days_since_any']   = last_by.min(axis=1)
pf = pf.reset_index()

# 3. attach to the scoring sheet; products never touched get 0 counts / window-length recency
cf = cand.merge(pf, on=['session_id', 'wid'], how='left')
cf['prod_seen_before'] = cf['prod_days_since_any'].notna().astype('int8')
for col in ['prod_n_click', 'prod_n_cart', 'prod_n_follow', 'prod_n_order']:
    cf[col] = cf[col].fillna(0).astype('int16')
for col, w in [('prod_days_since_click', 7), ('prod_days_since_cart', 90),
               ('prod_days_since_order', 365), ('prod_days_since_any', 365)]:
    cf[col] = cf[col].fillna(w).astype('float32')

print('Rows:', len(cf), '| Missing values:', cf.isna().sum().sum())
print('Share of rows where the person touched this product before: %.2f%%' % (100 * cf['prod_seen_before'].mean()))

# 4. first peek (TRAIN only): do previously-touched products get engaged more?
tr = cf[cf['split'] == 'train']
print('\nTrain rows: engagement and purchase rate, by touched-before:')
print(tr.groupby('prod_seen_before')['label'].agg(
    rows='size',
    engaged=lambda x: (x >= 1).mean(),
    purchased=lambda x: (x == 3).mean()).round(4))

Past actions on products in the current list: 145182
Rows: 7665446 | Missing values: 0
Share of rows where the person touched this product before: 0.78%

Train rows: engagement and purchase rate, by touched-before:
                     rows  engaged  purchased
prod_seen_before                             
0                 5319330   0.0191     0.0012
1                   41652   0.2488     0.0234


> **Result.** Only 0.78% of candidate rows were touched before, but those are engaged **24.9% vs 1.9%** (≈13×) and purchased **2.34% vs 0.12%** (≈19×) on the train split. The signal is rare but very precise; roughly 1 in 8 train purchases falls on a previously touched product.

### 5c. Save candidate-level features
One row per (search, candidate), keyed by `session_id` + `wid`. Includes `label` (target, not a feature) and `split`.

In [20]:
cf.to_parquet(OUT / 'history_candidate_features.parquet', index=False)
print('saved history_candidate_features.parquet')
print('Shape:', cf.shape)
print(cf.dtypes)

saved history_candidate_features.parquet
Shape: (7665446, 13)
session_id                 int32
wid                        int64
label                       int8
split                     object
prod_n_click               int16
prod_n_cart                int16
prod_n_follow              int16
prod_n_order               int16
prod_days_since_click    float32
prod_days_since_cart     float32
prod_days_since_order    float32
prod_days_since_any      float32
prod_seen_before            int8
dtype: object


## Step 6: User × brand / shop / category features
Exact-product history covers under 1% of candidates. Brands, shops and categories repeat far more, so we ask: *has the user interacted with this candidate's brand, shop or (most specific, level-4) category before?*

### 6a. Look up each product's brand, shop and category
The catalogue is read from `product_meta_data.txt` in 500k-row pieces, keeping only the needed columns and the products that appear in our histories or candidate lists. The shared `product_chunks` folder is empty, so it cannot be used.

In [27]:
needed = pd.unique(np.concatenate([hist['wid'].to_numpy(), cand['wid'].to_numpy()]))
print('Distinct products we need to look up:', len(needed))

parts = []
reader = pd.read_csv(BASE / 'product_meta_data.txt', sep='\t', dtype=str, chunksize=500_000,
                     usecols=['wid', 'brand_id', 'brand_name', 'shop_id', 'cate_id_4'])
for i, p in enumerate(reader):
    p['wid'] = pd.to_numeric(p['wid'], errors='coerce')
    p = p[p['wid'].isin(needed)].copy()                            # only products we need
    p['wid'] = p['wid'].astype('int64')
    p['has_brand_name'] = p['brand_name'].notna() & (p['brand_name'].str.len() > 0)
    parts.append(p.drop(columns='brand_name'))                     # drop the heavy text column
    if i % 5 == 0:
        print(f'  read {(i + 1) * 500_000:,} catalogue rows...')

cat = pd.concat(parts, ignore_index=True)
for col in ['brand_id', 'shop_id', 'cate_id_4']:
    cat[col] = pd.to_numeric(cat[col], errors='coerce')

print('Catalogue rows kept:', len(cat), '| each product once:', cat['wid'].is_unique)
print('Needed products found in catalogue: %.1f%%' % (100 * len(cat) / len(needed)))
print('History actions whose product is in catalogue: %.1f%%' % (100 * hist['wid'].isin(cat['wid']).mean()))
print('Candidate rows whose product is in catalogue: %.1f%%' % (100 * cand['wid'].isin(cat['wid']).mean()))
print('\nProducts with no brand name: %.1f%%' % (100 * (~cat['has_brand_name']).mean()))
print('Brand IDs used by products with no brand name (top 5):')
print(cat.loc[~cat['has_brand_name'], 'brand_id'].value_counts().head())
print('\nProducts with shop_id = -1: %.2f%%' % (100 * (cat['shop_id'] == -1).mean()))

Distinct products we need to look up: 7325823
  read 500,000 catalogue rows...
  read 3,000,000 catalogue rows...
  read 5,500,000 catalogue rows...
  read 8,000,000 catalogue rows...
  read 10,500,000 catalogue rows...
Catalogue rows kept: 6922295 | each product once: True
Needed products found in catalogue: 94.5%
History actions whose product is in catalogue: 95.1%
Candidate rows whose product is in catalogue: 99.5%

Products with no brand name: 6.7%
Brand IDs used by products with no brand name (top 5):
brand_id
38645246    253414
54289642    173453
98209701     40150
Name: count, dtype: int64

Products with shop_id = -1: 0.88%


> **Result.** 94.5% of needed products are in the catalogue, covering 95.1% of history actions and 99.5% of candidate rows (interim: 95.14% / 99.55%). Products with no brand name (6.7%) share just **3 placeholder brand IDs**, and 0.88% of products have shop `-1` (unknown). Both traps are handled in 6b.

### 6b. Clean brand / shop / category keys
A product's brand is treated as **unknown when it has no brand name**. We don't ban the 3 placeholder IDs directly, because the interim found one of them is also used by named products. Shop `-1` is unknown, and products missing from the catalogue get unknown keys. **Unknown keys never count as a match**, so two unbranded products are not treated as "same brand". History uses the same windows and the ≤1 s leakage guard as step 5.

In [28]:
cat['brand_key'] = cat['brand_id'].where(cat['has_brand_name'])     # no brand name -> unknown (NaN)
cat['shop_key']  = cat['shop_id'].where(cat['shop_id'] != -1)        # shop -1 -> unknown
cat['cate_key']  = cat['cate_id_4']
lookup = cat.set_index('wid')[['brand_key', 'shop_key', 'cate_key']]
KEYS = ['brand_key', 'shop_key', 'cate_key']

# history: in-window, leakage guard (ignore actions <=1 s before the search)
hg = inw.loc[inw['age_sec'] > 1, ['session_id', 'wid', 'type', 'age_sec', 'age_days']].copy()
cg = cand[['session_id', 'wid', 'label', 'split']].copy()
for k in KEYS:
    hg[k] = hg['wid'].map(lookup[k])
    cg[k] = cg['wid'].map(lookup[k])

print('Known key share:            history   candidates')
for k in KEYS:
    print(f'  {k:10s}               {hg[k].notna().mean():6.1%}     {cg[k].notna().mean():6.1%}')

print('\nShare of candidate rows where the user touched the SAME ... before:')
print(f'  exact product (step 5)  {0.0078:6.2%}')
for k in KEYS:
    pairs = hg[['session_id', k]].dropna().drop_duplicates()
    pairs['hit'] = 1
    hit = cg[['session_id', k]].merge(pairs, on=['session_id', k], how='left')['hit'].fillna(0)
    print(f'  {k:22s}  {hit.mean():6.2%}')

Known key share:            history   candidates
  brand_key                 93.5%      94.2%
  shop_key                  94.8%      98.6%
  cate_key                  95.1%      99.5%

Share of candidate rows where the user touched the SAME ... before:
  exact product (step 5)   0.78%
  brand_key               19.93%
  shop_key                 6.94%
  cate_key                37.11%


> **Result (reach).** Share of candidate rows where the user touched the same … before: exact product 0.78%, **shop 6.94%, brand 19.93%, category 37.11%**.

### 6c. Build the features
For each grouping (brand, shop, category): clicks (7 d), carts (90 d), orders counted as **distinct checkouts** (365 d), days since any action and days since the last order. That's 15 columns. Follows are omitted because they are too rare. Missing counts are 0; missing recency is 365.

In [29]:
def group_features(k, prefix):
    h = hg.dropna(subset=[k])                                         # unknown keys never match
    n = (h.groupby(['session_id', k, 'type'], observed=True).size()
          .unstack().reindex(columns=TYPES).fillna(0))
    ords = h[h['type'] == 'ORD']
    f = pd.DataFrame(index=n.index)
    f[f'{prefix}_n_click'] = n['CLICK']
    f[f'{prefix}_n_cart']  = n['CART']
    f[f'{prefix}_n_order'] = ords.drop_duplicates(['session_id', k, 'age_sec']).groupby(['session_id', k]).size()
    f[f'{prefix}_days_since_any']   = h.groupby(['session_id', k])['age_days'].min()
    f[f'{prefix}_days_since_order'] = ords.groupby(['session_id', k])['age_days'].min()
    return f.reset_index()

rows_before = len(cg)
for k, prefix in [('brand_key', 'brand'), ('shop_key', 'shop'), ('cate_key', 'cate')]:
    cg = cg.merge(group_features(k, prefix), on=['session_id', k], how='left')
    for col in [f'{prefix}_n_click', f'{prefix}_n_cart', f'{prefix}_n_order']:
        cg[col] = cg[col].fillna(0).astype('int32')
    for col in [f'{prefix}_days_since_any', f'{prefix}_days_since_order']:
        cg[col] = cg[col].fillna(365).astype('float32')
    print(f'{prefix}: done')

feat_cols = [c for c in cg.columns if c.startswith(('brand_', 'shop_', 'cate_')) and not c.endswith('_key')]
print('\nRows before/after:', rows_before, len(cg), '| new feature columns:', len(feat_cols))
print('Missing values in new features:', cg[feat_cols].isna().sum().sum())
for p in ['brand', 'shop', 'cate']:
    touched = (cg[f'{p}_days_since_any'] < 365).mean()
    print(f'Share of rows with any {p} history: {touched:.2%}')
display(cg[feat_cols].describe().T[['mean', '50%', 'max']].round(2))

brand: done
shop: done
cate: done

Rows before/after: 7665446 7665446 | new feature columns: 15
Missing values in new features: 0
Share of rows with any brand history: 19.93%
Share of rows with any shop history: 6.94%
Share of rows with any cate history: 37.11%


,mean,50%,max
brand_n_click,1.35,0.0,826.0
brand_n_cart,0.79,0.0,1206.0
brand_n_order,0.11,0.0,246.0
brand_days_since_any,296.46,365.0,365.0
brand_days_since_order,352.49,365.0,365.0
shop_n_click,0.30,0.0,689.0
shop_n_cart,0.16,0.0,517.0
shop_n_order,0.03,0.0,241.0
shop_days_since_any,341.31,365.0,365.0
shop_days_since_order,360.78,365.0,365.0


Save the brand / shop / category features: one row per (search, candidate), keyed by `session_id` + `wid`. Label and split are in the candidate file.

In [30]:
group_cols = ['session_id', 'wid'] + feat_cols
cg[group_cols].to_parquet(OUT / 'history_group_features.parquet', index=False)
print('saved history_group_features.parquet', cg[group_cols].shape)

saved history_group_features.parquet (7665446, 17)


## Step 7: Leakage check
If the current search's own interactions were logged at the end of history, previously touched **engaged** candidates would cluster a few seconds before the search. We compare when previously touched candidates were last touched, for engaged vs non-engaged candidates.

**Before the guard** (first run, without the ≤1 s filter): products last touched **≤1 s** before the search were 6.8% of engaged vs 0.7% of non-engaged previously touched candidates (1,047 vs 299 rows; ≈10× over-represented). One second is implausibly short for a person to issue a new search, and 4.57% of searches have their last action exactly 1 s before (interim: 4.6%). We cannot prove these are leaked actions, but excluding them costs almost nothing, so steps 5–6 ignore touches ≤1 s before the search.

The output below is **after** the guard, so the ≤1 s bucket is now empty. Some products moved into later buckets because their earlier touches now count as the most recent.

In [21]:
seen = cf[cf['prod_seen_before'] == 1].copy()
seen['secs'] = seen['prod_days_since_any'] * 86400          # last touched, in seconds before the search
seen['engaged'] = seen['label'] >= 1

bins  = [-1, 1, 60, 1800, 86400, 7 * 86400, 400 * 86400]
names = ['<=1 s', '1 s-1 min', '1-30 min', '30 min-1 day', '1-7 days', '>7 days']
seen['when'] = pd.cut(seen['secs'], bins, labels=names)

print('Touched-before products: when were they last touched? (counts)')
print(pd.crosstab(seen['when'], seen['engaged']))
print('\nSame, as share of each column:')
print(pd.crosstab(seen['when'], seen['engaged'], normalize='columns').round(3))

last_gap = (uf['days_since_last_action'] * 86400).round()
print('\nSearches whose last past action was exactly 1 s before: %.2f%%' % (100 * (last_gap == 1).mean()))

Touched-before products: when were they last touched? (counts)
engaged       False  True 
when                      
1 s-1 min      3789   2633
1-30 min      11823   4081
30 min-1 day   7372   2536
1-7 days      10504   2446
>7 days       11460   3144

Same, as share of each column:
engaged       False  True 
when                      
1 s-1 min     0.084  0.177
1-30 min      0.263  0.275
30 min-1 day  0.164  0.171
1-7 days      0.234  0.165
>7 days       0.255  0.212

Searches whose last past action was exactly 1 s before: 4.57%


Finer look within the first minute: is there a second cliff just after 1 s, which would justify a wider guard?

In [22]:
s60 = cf[cf['prod_seen_before'] == 1].copy()
s60['secs'] = s60['prod_days_since_any'] * 86400
s60['engaged'] = s60['label'] >= 1
s60 = s60[s60['secs'] <= 60]
s60['when'] = pd.cut(s60['secs'], [1, 5, 15, 30, 60], labels=['1-5 s', '5-15 s', '15-30 s', '30-60 s'])
ct = pd.crosstab(s60['when'], s60['engaged'])
ct['engaged_rate'] = (ct[True] / (ct[True] + ct[False])).round(3)
print(ct)

engaged  False  True  engaged_rate
when                              
1-5 s       85   178         0.677
5-15 s     443   580         0.567
15-30 s   1170   790         0.403
30-60 s   2091  1085         0.342


> **Result.** Beyond 1 s, the engaged rate declines **smoothly** with time since the last touch: 0.68 (1–5 s), 0.57, 0.40, 0.34, then ≈0.26 (1–30 min) down to ≈0.19–0.22 after a day. This is a genuine recency effect, not a cliff, so the guard is kept at 1 s. Most engaged previously touched products were last touched minutes to weeks earlier, so there is no wholesale leak.

## Step 8: Evidence of feature effectiveness (train split only)
Descriptive evidence that the history features carry signal: (1) engagement by recency of the last touch on the product, (2) by type of past action on the product, (3) the interaction between user context (`in_session`) and product history, and (4) user-level features vs whether the search ends in a purchase. The validation and test splits are not used here. The final evidence is the LambdaRank ablation (with vs without these features).

In [23]:
tr = cf[cf['split'] == 'train'].copy()
tr['engaged']   = tr['label'] >= 1
tr['purchased'] = tr['label'] == 3

def rates(df, by):
    return df.groupby(by, observed=True).agg(rows=('engaged', 'size'),
                                             engaged_rate=('engaged', 'mean'),
                                             purchase_rate=('purchased', 'mean')).round(4)

# 1. Recency curve: how long ago was this product last touched?
labels = ['1 s-1 min', '1-30 min', '30 min-1 day', '1-7 days', '7-90 days', '90-365 days']
tr['last_touched'] = pd.cut(tr['prod_days_since_any'], [-1, 1/1440, 30/1440, 1, 7, 90, 366], labels=labels).astype(str)
tr.loc[tr['prod_seen_before'] == 0, 'last_touched'] = 'never touched'
print('1. By how long ago the person last touched THIS product')
display(rates(tr, 'last_touched').reindex(labels + ['never touched']))

# 2. Which kind of past action on this product?
rows = []
for name, mask in [('never touched',     tr['prod_seen_before'] == 0),
                   ('clicked it (7d)',   tr['prod_n_click']  > 0),
                   ('carted it (90d)',   tr['prod_n_cart']   > 0),
                   ('followed it (90d)', tr['prod_n_follow'] > 0),
                   ('ordered it (365d)', tr['prod_n_order']  > 0)]:
    sub = tr[mask]
    rows.append((name, len(sub), sub['engaged'].mean(), sub['purchased'].mean()))
print('2. By type of past action on THIS product (a product can be in several rows)')
display(pd.DataFrame(rows, columns=['past action', 'rows', 'engaged_rate', 'purchase_rate']).round(4))

# 3. Does user context change how much product history matters?
tr['in_session'] = tr['session_id'].map(uf['in_session'])
print('3. Touched-before products only, by whether the user is mid-visit')
display(rates(tr[tr['prod_seen_before'] == 1], 'in_session'))

# 4. User-level features vs whether the search ends in a purchase
ut = uf[uf['split'] == 'train'].join(work.set_index('session_id')['has_purchase'])
print('4a. Search ends in purchase, by in_session')
display(ut.groupby('in_session')['has_purchase'].agg(['size', 'mean']).round(4))
print('4b. ...by checkouts in the last year')
display(ut.groupby(pd.cut(ut['n_checkouts_365d'], [-1, 0, 5, 20, 100, 1e6],
                          labels=['0', '1-5', '6-20', '21-100', '100+']), observed=True)['has_purchase'].agg(['size', 'mean']).round(4))
print('4c. ...by days since last cart')
display(ut.groupby(pd.cut(ut['days_since_last_cart'], [-1, 1, 7, 30, 89.99, 90],
                          labels=['<1 day', '1-7 days', '7-30 days', '30-90 days', 'no cart in 90 d']), observed=True)['has_purchase'].agg(['size', 'mean']).round(4))

1. By how long ago the person last touched THIS product


,rows,engaged_rate,purchase_rate
last_touched,,,
1 s-1 min,4395,0.4146,0.0414
1-30 min,11025,0.2595,0.0205
30 min-1 day,6863,0.2592,0.0166
1-7 days,9037,0.1884,0.0096
7-90 days,8204,0.2087,0.0256
90-365 days,2128,0.2284,0.0728
never touched,5319330,0.0191,0.0012


2. By type of past action on THIS product (a product can be in several rows)


,past action,rows,engaged_rate,purchase_rate
0,never touched,5319330,0.0191,0.0012
1,clicked it (7d),29364,0.2571,0.0178
2,carted it (90d),18116,0.2381,0.0275
3,followed it (90d),19,0.2105,0.0000
4,ordered it (365d),4611,0.2574,0.0878


3. Touched-before products only, by whether the user is mid-visit


,rows,engaged_rate,purchase_rate
in_session,,,
0,12120,0.2361,0.0290
1,29532,0.2540,0.0211


... (2 outputs left out)

4b. ...by checkouts in the last year


,size,mean
n_checkouts_365d,,
0,5116,0.0452
1-5,14982,0.0830
6-20,18803,0.1135
21-100,18958,0.1586
100+,2245,0.2192


4c. ...by days since last cart


,size,mean
days_since_last_cart,,
<1 day,23315,0.1215
1-7 days,19307,0.1180
7-30 days,11085,0.1095
30-90 days,3771,0.1013
no cart in 90 d,2626,0.1523


> **Results**
> 1. **Recency.** Every "touched" bucket is engaged 10–20× more than never-touched products (1.9%). Engagement falls over the first week (41% → 19%), then levels off. **Purchase is U-shaped** (4.1% → 1.0% at 1–7 days → **7.3% at 90–365 days**). Touches older than 7 days can only be carts or orders, so the late rise is repeat buying. This is why per-type recency is kept, not just `prod_days_since_any`.
> 2. **"Buy it again."** Previously **ordered** products are re-bought at **8.8%** (≈73× baseline), compared with 2.75% for carted and 1.8% for clicked. Follows are too rare to judge (19 rows).
> 3. **Context.** Being mid-visit changes product-history rates only slightly, and in opposite directions (engaged 25.4% vs 23.6%; purchased 2.1% vs 2.9%), so no strong context effect is claimed.
> 4. **User level.** The purchase probability of a search rises steadily with checkouts in the last year (0 → 4.5%, 100+ → 21.9%; ≈5×). Mid-visit searches end in purchase slightly less often (10.6% vs 12.6%). Recency of the last cart has a mild effect, and users with no cart in 90 days buy more often (15.2%, n = 2,626), which is a hypothesis to examine in diagnostics.

### 8.5. Brand / shop / category history (step 6)
Rates by whether the user has history with the candidate's brand, shop or category. A feature can only reorder a list if it **differs between candidates in that list**, so we also report how often it does, and the engagement gap within those lists.

In [31]:
trg = cg[cg['split'] == 'train'].copy()
trg['engaged']   = trg['label'] >= 1
trg['purchased'] = trg['label'] == 3

rows = []
for p in ['brand', 'shop', 'cate']:
    touched = trg[f'{p}_days_since_any'] < 365
    ordered = trg[f'{p}_n_order'] > 0
    share_on = touched.groupby(trg['session_id']).mean()
    mixed_sessions = share_on[(share_on > 0) & (share_on < 1)].index        # lists where it differs
    in_mixed = trg['session_id'].isin(mixed_sessions)
    rows.append({
        'group': p,
        'rows touched':                 int(touched.sum()),
        'engaged | touched':            trg.loc[touched, 'engaged'].mean(),
        'engaged | not':                trg.loc[~touched, 'engaged'].mean(),
        'purchased | touched':          trg.loc[touched, 'purchased'].mean(),
        'purchased | not':              trg.loc[~touched, 'purchased'].mean(),
        'purchased | ordered from':     trg.loc[ordered, 'purchased'].mean(),
        'lists where it differs':       len(mixed_sessions) / trg['session_id'].nunique(),
        'in those lists: engaged | touched': trg.loc[in_mixed & touched, 'engaged'].mean(),
        'in those lists: engaged | not':     trg.loc[in_mixed & ~touched, 'engaged'].mean(),
    })
pd.set_option('display.max_columns', 20)
display(pd.DataFrame(rows).set_index('group').round(4))

group,brand,shop,cate
rows touched,1.064887e+06,372051.0000,1.982362e+06
engaged | touched,3.170000e-02,0.0644,2.590000e-02
engaged | not,1.820000e-02,0.0176,1.800000e-02
purchased | touched,2.100000e-03,0.0052,1.500000e-03
purchased | not,1.200000e-03,0.0011,1.300000e-03
purchased | ordered from,4.500000e-03,0.0119,2.800000e-03
lists where it differs,5.223000e-01,0.5009,5.392000e-01
in those lists: engaged | touched,3.180000e-02,0.0669,2.550000e-02
in those lists: engaged | not,1.560000e-02,0.0145,1.370000e-02


> **Result.** **Shop** history is the strongest group signal: engaged 6.4% vs 1.8% (≈3.7×), purchased 0.52% vs 0.11% (≈4.7×), and 1.19% purchase if the user ordered from that shop before (≈10× baseline). **Brand** is moderate (≈1.7×) with wide reach (20% of rows). **Category** has the widest reach (37%) but is the weakest, barely moving purchase (0.15% vs 0.13%). All three differ between candidates in about half of the lists, and within those lists touched candidates are engaged 2.0× (brand), 4.6× (shop) and 1.9× (category) more. Together with step 5, this gives a **reach vs precision** trade-off: exact product is rare but very strong, while shop, brand and category are broader but weaker.

## Step 9: Final checks and feature dictionary
Reload the saved files from Drive, verify keys, row counts and missing values, and save a feature dictionary describing every column.

In [24]:
u = pd.read_parquet(OUT / 'history_user_features.parquet')
c = pd.read_parquet(OUT / 'history_candidate_features.parquet')

print('USER file:', u.shape, '| one row per search:', u['session_id'].is_unique, '| missing:', u.isna().sum().sum())
print('CANDIDATE file:', c.shape, '| (session, wid) unique:', not c.duplicated(['session_id', 'wid']).any(),
      '| missing:', c.isna().sum().sum())
print('Every candidate search has user features:', c['session_id'].isin(u['session_id']).all())
print('Splits agree between files:',
      (c.drop_duplicates('session_id').set_index('session_id')['split'] == u.set_index('session_id')['split']).all())

D = [  # (feature, file, meaning, window, if none)
 ('session_id', 'both', 'Search ID = row number in user_subset.parquet (join key)', '-', '-'),
 ('wid', 'candidate', 'Candidate product ID, int64 (join key)', '-', '-'),
 ('label', 'candidate', 'TARGET, not a feature: 0 none, 1 click, 2 cart, 3 purchase', '-', '-'),
 ('split', 'both', 'train / val / test (from user_subset.parquet)', '-', '-'),
 ('n_click_7d', 'user', 'Clicks in the last 7 days', '7 d', '0'),
 ('n_cart_90d', 'user', 'Add-to-carts in the last 90 days', '90 d', '0'),
 ('n_follow_90d', 'user', 'Follows in the last 90 days', '90 d', '0'),
 ('n_order_rows_365d', 'user', 'Order rows in the last year (before merging checkouts)', '365 d', '0'),
 ('n_checkouts_365d', 'user', 'Checkouts in the last year (same-second orders merged)', '365 d', '0'),
 ('n_order_products_365d', 'user', 'Distinct products ordered in the last year', '365 d', '0'),
 ('has_click / has_cart / has_order', 'user', '1 if at least one such action within its window', '7 / 90 / 365 d', '0'),
 ('days_since_last_click', 'user', 'Days since most recent click', '7 d', '7 (= at least 7)'),
 ('days_since_last_cart', 'user', 'Days since most recent add-to-cart', '90 d', '90'),
 ('days_since_last_order', 'user', 'Days since most recent order', '365 d', '365'),
 ('days_since_last_action', 'user', 'Days since most recent action of any type', 'all', 'never missing'),
 ('in_session', 'user', '1 if last action was within 30 min of the search (ongoing visit)', '30 min', '-'),
 ('n_actions_7d', 'user', 'All actions in the last 7 days', '7 d', '0'),
 ('intent_share_7d', 'user', 'Share of last-7-day actions that are carts or orders', '7 d', '0 (check n_actions_7d)'),
 ('history_len_full', 'user', 'Total past actions in the restored full history', 'all', '-'),
 ('prod_n_click', 'candidate', 'Times the user clicked THIS product', '7 d', '0'),
 ('prod_n_cart', 'candidate', 'Times the user carted THIS product', '90 d', '0'),
 ('prod_n_follow', 'candidate', 'Times the user followed THIS product', '90 d', '0'),
 ('prod_n_order', 'candidate', 'Times the user ordered THIS product', '365 d', '0'),
 ('prod_days_since_click', 'candidate', 'Days since the user last clicked THIS product', '7 d', '7'),
 ('prod_days_since_cart', 'candidate', 'Days since the user last carted THIS product', '90 d', '90'),
 ('prod_days_since_order', 'candidate', 'Days since the user last ordered THIS product', '365 d', '365'),
 ('prod_days_since_any', 'candidate', 'Days since any action on THIS product', '365 d', '365'),
 ('prod_seen_before', 'candidate', '1 if the user touched THIS product before (any type)', '365 d', '0'),
]
fdict = pd.DataFrame(D, columns=['feature', 'file', 'meaning', 'window', 'if none in window'])
fdict.to_csv(OUT / 'feature_dictionary.csv', index=False)
print('\nNote: product-level features ignore history actions <=1 s before the search (leakage guard).')
pd.set_option('display.max_colwidth', 80)
display(fdict)

USER file: (85864, 19) | one row per search: True | missing: 0
CANDIDATE file: (7665446, 13) | (session, wid) unique: True | missing: 0
Every candidate search has user features: True
Splits agree between files: True

Note: product-level features ignore history actions <=1 s before the search (leakage guard).


,feature,file,meaning,window,if none in window
0,session_id,both,Search ID = row number in user_subset.parquet (join key),-,-
1,wid,candidate,"Candidate product ID, int64 (join key)",-,-
2,label,candidate,"TARGET, not a feature: 0 none, 1 click, 2 cart, 3 purchase",-,-
3,split,both,train / val / test (from user_subset.parquet),-,-
4,n_click_7d,user,Clicks in the last 7 days,7 d,0
5,n_cart_90d,user,Add-to-carts in the last 90 days,90 d,0
6,n_follow_90d,user,Follows in the last 90 days,90 d,0
7,n_order_rows_365d,user,Order rows in the last year (before merging checkouts),365 d,0
8,n_checkouts_365d,user,Checkouts in the last year (same-second orders merged),365 d,0
9,n_order_products_365d,user,Distinct products ordered in the last year,365 d,0


Add the step 6 (brand / shop / category) features to the dictionary and check the group file.

In [33]:
new_rows = []
for p, name in [('brand', 'brand'), ('shop', 'shop'), ('cate', 'most specific category (level 4)')]:
    new_rows += [
        (f'{p}_n_click', 'group', f'Clicks on products of THIS candidate\'s {name}', '7 d', '0'),
        (f'{p}_n_cart', 'group', f'Carts on products of THIS candidate\'s {name}', '90 d', '0'),
        (f'{p}_n_order', 'group', f'Separate checkouts containing THIS candidate\'s {name}', '365 d', '0'),
        (f'{p}_days_since_any', 'group', f'Days since any action on THIS candidate\'s {name}', '365 d', '365'),
        (f'{p}_days_since_order', 'group', f'Days since last order from THIS candidate\'s {name}', '365 d', '365'),
    ]
fdict = fdict[fdict['file'] != 'group']                                # avoid duplicates if re-run
fdict = pd.concat([fdict, pd.DataFrame(new_rows, columns=fdict.columns)], ignore_index=True)
fdict.to_csv(OUT / 'feature_dictionary.csv', index=False)

g = pd.read_parquet(OUT / 'history_group_features.parquet')
print('GROUP file:', g.shape, '| (session, wid) unique:', not g.duplicated(['session_id', 'wid']).any(),
      '| missing:', g.isna().sum().sum())
print('Same rows as candidate file:', len(g) == len(cand))
print('Dictionary entries now:', len(fdict))
print('Note: unknown brand = no brand name; shop -1 = unknown; unknown keys never match; touches <=1 s before search ignored.')

GROUP file: (7665446, 17) | (session, wid) unique: True | missing: 0
Same rows as candidate file: True
Dictionary entries now: 43
Note: unknown brand = no brand name; shop -1 = unknown; unknown keys never match; touches <=1 s before search ignored.


## Handoff
All files are in `BASE/step1a_history/`:

| File | Rows | Key | Contents |
|---|---|---|---|
| `history_user_features.parquet` | 85,864 | `session_id` | `split` + 17 user-level features |
| `history_candidate_features.parquet` | 7,665,446 | `session_id` + `wid` | `label` (target), `split` + 9 exact-product features |
| `history_group_features.parquet` | 7,665,446 | `session_id` + `wid` | 15 brand / shop / category features |
| `feature_dictionary.csv` | 43 entries | – | Meaning, window and fill rule for every column |
| `history_full.parquet`, `history_long.parquet` | – | – | Intermediate tables (full histories; one row per past action) |

**For the ranker:** join the candidate and group files on `session_id` + `wid`, and the user file on `session_id`. Never use `label` as a feature. Compute any cross-user statistics on the train split only.